# 01 — Exploratory Data Analysis

Five rubric-required charts on the per-page feature matrix `data/processed/features.csv`. Every chart is exported to `assets/charts/` at ≥300 DPI and is annotated with the modeling decision it informs.

**Why these five:** class balance drives metric choice (F1/PR-AUC over accuracy); feature distributions drive scaling vs tree-friendly models; correlation heatmap tells us whether linear models can plausibly compete; PageRank distribution justifies the graph-feature inclusion; TF-IDF top terms confirm topical signal in our content features.

Pipeline prerequisite: `python -m src.scraping.doc_scraper` → `... serp_client build-queries` → `... fetch` → `... build_features` → `python -m src.graph.build_graph` → `python -m src.graph.graph_features`.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
CHART_DIR = Path("../assets/charts"); CHART_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv("../data/processed/features.csv")
print(f"rows={len(df)}  cols={df.shape[1]}")
df.head()

## Chart 1 — Class balance

**Modeling decision informed:** picking F1 + PR-AUC as the primary metrics (not accuracy). With imbalance > ~3:1, accuracy rewards a model that always predicts the majority class.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5), dpi=300)
counts = df["is_top_10"].value_counts().sort_index()
labels = ["Not top-10", "Top-10"]
bars = ax.bar(labels, counts.values, color=["#94a3b8", "#4f46e5"])
for b, v in zip(bars, counts.values):
    ax.text(b.get_x() + b.get_width()/2, v, f" {v} ({v/len(df)*100:.1f}%)", ha="center", va="bottom", fontsize=10)
ax.set_ylabel("page count"); ax.set_title("Class balance — is_top_10"); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.savefig(CHART_DIR / "01_class_balance.png", dpi=300, bbox_inches="tight"); plt.show()

## Chart 2 — Feature distributions by class

**Modeling decision informed:** which features are well-scaled (LogReg-friendly) vs heavy-tailed (tree-friendly). Strong per-class separation here = signal we can defend; flat overlap = the feature mostly adds noise.

In [ ]:
feat_cols = [c for c in ["title_length","word_count","keyword_density","h2_count","alt_text_coverage","pagerank"] if c in df.columns]
fig, axes = plt.subplots(2, 3, figsize=(14, 8), dpi=300)
for ax, col in zip(axes.ravel(), feat_cols):
    sns.kdeplot(data=df, x=col, hue="is_top_10", common_norm=False, ax=ax, palette={0:"#94a3b8", 1:"#4f46e5"}, fill=True, alpha=0.35, linewidth=1.5)
    ax.set_title(col); ax.set_ylabel(""); ax.spines[["top","right"]].set_visible(False)
plt.suptitle("Per-feature distributions by class", y=1.02, fontsize=14, fontweight="bold")
plt.tight_layout(); plt.savefig(CHART_DIR / "02_feature_distributions.png", dpi=300, bbox_inches="tight"); plt.show()

## Chart 3 — Correlation heatmap

**Modeling decision informed:** if features are highly correlated, regularization (L1 / L2) helps; if they are largely independent, ensembles (RF / XGB) gain less from shrinkage but benefit from feature interactions.

In [ ]:
non_feat = {"url","domain","query_id","query"}
num_df = df.drop(columns=[c for c in non_feat if c in df.columns]).select_dtypes(include=[np.number])
core_cols = [c for c in num_df.columns if not c.startswith("tfidf_")]
corr = num_df[core_cols].corr()
fig, ax = plt.subplots(figsize=(12, 10), dpi=300)
sns.heatmap(corr, cmap="RdBu_r", center=0, vmin=-1, vmax=1, square=True, cbar_kws={"shrink":0.7}, ax=ax, linewidths=0.3)
ax.set_title("Feature correlation (TF-IDF columns excluded for legibility)")
plt.tight_layout(); plt.savefig(CHART_DIR / "03_correlation_heatmap.png", dpi=300, bbox_inches="tight"); plt.show()

## Chart 4 — PageRank distribution (log scale)

**Modeling decision informed:** confirms graph features carry signal (long-tailed PageRank ≠ uniform). If PageRank were uniform, including it in the model would add noise without information.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
sns.histplot(df["pagerank"].clip(lower=1e-6), bins=40, log_scale=(True, False), color="#4f46e5", edgecolor="white", ax=ax)
ax.set_xlabel("PageRank (log scale)"); ax.set_ylabel("page count"); ax.set_title("PageRank distribution across the corpus")
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.savefig(CHART_DIR / "04_pagerank_distribution.png", dpi=300, bbox_inches="tight"); plt.show()

## Chart 5 — TF-IDF top terms by class

**Modeling decision informed:** if top-10 pages have visibly different topical vocabulary than non-top-10 pages on the same query, the TF-IDF columns should help; if not, we'd prune them.

In [ ]:
tfidf_cols = [c for c in df.columns if c.startswith("tfidf_")]
if tfidf_cols:
    means = df.groupby("is_top_10")[tfidf_cols].mean().T
    means.columns = ["not_top_10", "top_10"] if list(means.columns) == [0,1] else means.columns.astype(str)
    means["diff"] = means.iloc[:,1] - means.iloc[:,0]
    top_terms = means.reindex(means["diff"].abs().sort_values(ascending=False).index).head(20)
    fig, ax = plt.subplots(figsize=(10, 8), dpi=300)
    top_terms[["not_top_10","top_10"]].plot(kind="barh", ax=ax, color=["#94a3b8","#4f46e5"])
    ax.set_xlabel("mean TF-IDF score"); ax.set_title("Top-20 TF-IDF terms — mean score by class")
    ax.invert_yaxis(); ax.spines[["top","right"]].set_visible(False)
    plt.tight_layout(); plt.savefig(CHART_DIR / "05_tfidf_top_terms.png", dpi=300, bbox_inches="tight"); plt.show()
else:
    print("no tfidf_* columns found — re-run src.features.build_features")